```mermaid
flowchart LR

    %% =========================
    %% Plain Net
    %% =========================
    subgraph Plain["Plain net"]
        direction TB

        X1["x"]
        W1["Weight Layer"]
        Relu1["relu"]
        W2["Weight Layer"]
        Relu2["relu"]
        H1["H(x)"]

        X1 --> W1
        W1 --> Relu1
        Relu1 --> W2
        W2 --> Relu2
        Relu2 --> H1
    end


    %% =========================
    %% Residual Net
    %% =========================
    subgraph Residual["Residual net"]
        direction TB

        X2["x"]
        W3["Weight Layer"]
        Relu3["relu"]
        W4["Weight Layer"]
        Add(("＋"))
        Relu4["relu"]
        H2["H(x)"]

        %% 主路径
        X2 --> W3
        W3 --> Relu3
        Relu3 --> W4
        W4 -->|"F(x)"| Add
        Add --> Relu4
        Relu4 --> H2

        %% Shortcut / identity mapping
        X2 -->|"x"| Add
    end
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 定义残差块
class Resblock(nn.Module):
    def __init__(self, input_channels):
        super(Resblock, self).__init__()
        self.input_channels = input_channels
        self.conv1 = nn.Conv2d(input_channels, input_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(input_channels, input_channels, kernel_size=3, padding=1)
    def forward(self, x):
        residual = x
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        x += residual
        x = F.relu(x)
        return x
#构建残差神经网络
class ResNet(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.resblock1 = Resblock(64)
        self.resblock2 = Resblock(64)
        self.pool2 = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = self.resblock1(x)
        x = self.resblock2(x)
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x